# Current v3.1 confusion matrices

Rows are GT interval labels and columns are the labels with greatest accumulated predicted overlap. `<no_overlap>` is a temporal miss. These are micro interval counts; they are not the macro-per-video F1 values. Use `normalized=True` for within-GT-class rates.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
archive_dir = here if (here / 'results' / 'current').exists() else here / 'notebooks' / 'archive'
matrix_dir = archive_dir / 'results' / 'current' / 'confusion_matrices'
summary = pd.read_csv(matrix_dir / 'confusion_summary.csv')
pairs = pd.read_csv(matrix_dir / 'confusion_pairs_long.csv')
display(summary.round(3))

In [ ]:
def show_confusion(actor, level, normalized=True):
    suffix = 'row_normalized' if normalized else 'counts'
    matrix = pd.read_csv(matrix_dir / f'{actor}_{level}_{suffix}.csv', index_col='gt_label')
    width = min(24, max(7, 0.55 * len(matrix.columns)))
    height = min(30, max(4, 0.42 * len(matrix.index)))
    fig, ax = plt.subplots(figsize=(width, height))
    image = ax.imshow(matrix.values, aspect='auto', cmap='Blues', vmin=0)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(len(matrix.index)), matrix.index, fontsize=8)
    ax.set_xlabel('Predicted max-overlap label')
    ax.set_ylabel('GT label')
    title_suffix = suffix.replace('_', ' ')
    ax.set_title(f'v3.1 {actor} {level} — {title_suffix}')
    if matrix.size <= 180:
        for row in range(len(matrix.index)):
            for col in range(len(matrix.columns)):
                value = matrix.iat[row, col]
                text = f'{value:.2f}' if normalized else f'{int(value)}'
                ax.text(col, row, text, ha='center', va='center', fontsize=7)
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    return matrix

## Separate actor matrices at the release-score levels

Change the level argument to `raw`, `medium`, or `coarse` to inspect any of the nine complete matrices.

In [ ]:
left_raw = show_confusion('left', 'raw', normalized=True)

In [ ]:
right_medium = show_confusion('right', 'medium', normalized=True)

In [ ]:
camera_coarse = show_confusion('camera', 'coarse', normalized=True)

In [ ]:
# Largest non-matching cells. Right-raw matches can be off-diagonal because
# its optional contexts use partial unordered overlap rather than equality.
errors = pairs[pairs.evaluator_match_count == 0].sort_values(['actor', 'level', 'count'], ascending=[True, True, False])
display(errors.groupby(['actor', 'level'], group_keys=False).head(10))